# EDA (Exploratory Data Analysis) - Complete Workflow Notes

## Overall ML Pipeline
1. EDA (Exploratory Data Analysis)
2. Data profiling
3. Feature engineering (remove unnecessary columns)
4. Feature selection: get features that contribute to the ML model
5. Model Finalization

Core goal: Identify patterns in the data, and then train the model

---

## Supervised Model
- Input features (X) → Output feature / label (y)
- The model needs to be given the "answer" (label) so it can learn the mapping from X → y

### Handling text data
- "Text data" needs to be converted into numerical form before feeding it to an ML model
- Methods: Label Encoding / One-Hot Encoding, or simply drop columns that aren't needed

---

## Data Profiling

### `.info(verbose=True)`
- Shows data type, non-null count, and memory usage for each column

### `.describe()`
- Only works on **numerical** columns, ignores NaN values
- Output: count / mean / std / min / 25% / 50% / 75% / max
- `.describe().T` → transpose, swaps rows and columns, easier to read (especially with many columns)

Meaning of each statistic:
- count: number of non-null values
- mean: average value
- std: Standard Deviation
- min/max: minimum/maximum value
- 25%/50%/75%: percentiles/quartiles, useful for detecting outliers

---

## Handling Missing Values / Outliers

### Finding null values
```python
df.isnull().sum()
```
> Note: In some datasets, missing values aren't represented as NaN but as 0 (e.g. in the Diabetes dataset, Glucose/BloodPressure = 0 doesn't make sense — it's really a missing value). These need to be manually replaced with NaN first:
```python
df[cols] = df[cols].replace(0, np.nan)
```

### Deciding how to fill missing values
- Plot a histogram `hist()` first to check the distribution
- **Skewness** helps determine whether the distribution is symmetric or has extreme outliers

### Filling rules
| Situation | Fill method |
|---|---|
| Histogram looks roughly normal (symmetric) | Fill with **mean** |
| Skewed distribution / has extreme outliers | Fill with **median**, since median is not sensitive to extreme values |

```python
df['col'].fillna(df['col'].mean(), inplace=True)   # normal distribution
df['col'].fillna(df['col'].median(), inplace=True) # skewed / has outliers
```

### Anomaly
- Should be identified before filling missing values, otherwise outliers can distort the representativeness of statistics like the mean

---

## Correlation Analysis

- **Multicollinearity**: when multiple independent variables are highly correlated with each other, which can affect model stability/interpretability — needs to be checked
- **Pearson's Correlation**: measures the strength of the **linear** relationship between two variables, ranges from -1 to 1
  - 1: strong positive correlation, -1: strong negative correlation, 0: no correlation
- Visualization: `sns.heatmap(df.corr(), annot=True, cmap='RdYlGn')`

### Ideal situation
- Input and Output should ideally have a **linear relationship**
- Example: both age and experience may affect salary, but if age and experience are highly correlated (collinearity), it's usually better to **keep only experience and drop age**, to avoid redundant features

### Standardization / Z-score
- **Z-score** is used to bring variables with different scales onto a common standard, making it possible to **compare different things**
- Commonly done with `StandardScaler()` on the features (excluding the label column)

---

## Model Training

### Splitting the dataset
- **Seen data (Training Set)**: data the model learns from
- **Unseen data (Test Set)**: used to evaluate the model's real-world performance, avoiding the model just "memorizing answers"

# Zomato Dataset Notes

- Data characteristic: **majority of the data is text**
- Core question: which features are actually useful? Which ones genuinely contribute to the analysis goal (e.g. rating, location selection, etc.)?

### Standard cleaning steps
1. `.info()` to check overall structure (column types, nulls, memory usage)
2. Drop unnecessary columns (e.g. url, phone, dish_liked — columns that don't directly help the analysis)
```python
   df = df.drop(['url','dish_liked','phone'], axis=1)
```
3. Check and handle **duplicates**
```python
   df.duplicated().sum()
   df.drop_duplicates(inplace=True)
```
4. **Normalize column names**: if column names contain `()`, spaces, or special characters, it's better to standardize them (e.g. lowercase + underscores) so they can be accessed directly via `df.column_name`
```python
   df.columns = df.columns.str.replace('[()]', '', regex=True).str.strip().str.replace(' ', '_')
```

### Business Problem
Goal: understand the factors affecting the establishment of different types of restaurants in different locations in Bengaluru, and their aggregate rating, including:
- Cost of Restaurant
- Number of restaurants in a Location
- Restaurant type
- Most famous restaurant chains in Bengaluru